# Calculate Depression Depths and Geometries per Subbasin

## Contents
- [Input Datasets for this Notebook](#input-datasets-for-this-notebook)
- [0. Import required modules](#0-import-required-modules)
- [1. Organize data folders and files](#1-organize-data-folders-and-files)
    - [1.1 Create "DepressionDepths" subfolder](#11-create-depressiondepths-subfolder)
    - [1.2 Create subfolders in "DepressionDepths" subfolder ("Raw", "Interim", "Processed")](#12-create-subfolders-in-depressiondepths-subfolder-raw-interim-processed)
    - [1.3 Move input data files to "Raw" subfolder](#13-move-input-data-files-to-raw-subfolder)
- [2. Declare input data file names as variables](#2-declare-input-data-file-names-as-variables)
- [3. Read CLRH subbasins shapefile](#3-read-clrh-subbasins-shapefile)
- [4. Merge rasters if multiple LiDAR rasters are provided](#4-merge-rasters-if-multiple-lidar-rasters-provided)
    - [4.1 Clip each raster to CLRH subbasin shapefile for memory efficiency](#41-clip-each-raster-to-clrh-subbasins-shapefile-for-memory-efficiency)
    - [4.2 Find the CRS for the highest resoltion raster](#42-find-the-crs-for-the-highest-resolution-raster)
    - [4.3 Project all rasters to the CRS of the highest resolution raster](#43-project-all-rasters-to-the-crs-of-the-highest-resolution-raster)
    - [4.4 Merge all rasters together](#44-merge-all-rasters-together)
    - [4.5 Fill in gaps in merged raster](#45-fill-in-gaps-in-merged-raster)
- [5. Process LiDAR raster DEM](#5-process-lidar-raster-dem)
    - [5.1 Project subbasins shapefile to align with LiDAR raster DEM](#51-project-subbasins-shapefile-to-align-with-lidar-raster-dem)
    - [5.2 Clip LiDAR raster DEM to watershed](#52-clip-lidar-raster-dem-to-watershed)
    - [5.3 Resample clipped LiDAR raster DEM to 5m resolution](#53-resample-clipped-lidar-raster-dem-to-5m-resolution)
- [6. Process NHN Streams Network Shapefile](#6-process-nhn-streams-network-shapefile)
    - [6.1 Project subbasins shapefile to align with NHN streams shapefile](#61-project-subbasins-shapefile-to-align-with-nhn-streams-shapefile)
    - [6.2 Clip NHN streams shapefile to watershed](#62-clip-nhn-streams-shapefile-to-watershed)
    - [6.3 Visualize clipped and projected NHN streams shapefile](#63-visualize-clipped-and-projected-nhn-streams-shapefile)
- [7. Generate depressions raster](#7-generate-depressions-raster)
    - [7.1 Generate depressions raster](#71-generate-depressions-raster)
    - [7.2 Generate Wetlands Shapefile with Stats (this is optional for the user and not needed for RAVEN)](#72-generate-wetlands-shapefile-with-stats-this-is-optional-for-the-user-and-not-needed-for-raven)
- [8. Calculate Depression Depths](#8-calculate-depression-depths)
    - [8.1 Calculate Subbasin Zonal Statistics](#81-calculate-subbasin-zonal-statistics)
    - [8.2 Calculate depression depths from zonal statistics](#82-calculate-depression-depths-from-zonal-statistics)
    - [8.3 Write depression depths to file](#83-write-depression-depths-to-file)
- [9. Visualize Processed Data](#9-visualize-processed-data)
- [10. Calculate basin aspect ratios from geometry](#10-calculate-basin-aspect-ratios-from-geometry)








## Input Datasets for this Notebook

Prior to running this notebook, the following zip files must be downloaded and extracted into a new project folder called "Data" at the root project directory:
- Watershed of interest based on outlet point from the [CLRH Hydrofabrics](https://hydrology.uwaterloo.ca/CLRH/Hydrofabric.html) 
    - e.g., ```ID: 05OE006``` for Manning Canal
- Streams dataset of interest from [NHN streams](https://ftp.maps.canada.ca/pub/nrcan_rncan/vector/geobase_nhn_rhn/shp_en/)
    - e.g., ```ngn_rhn_05oe000_shp_en.zip``` for Manning Canal
- Raster DEM(s) of interest from [LiDAR DEMs](https://mli.gov.mb.ca/dems/index_external_lidar.html)
    - e.g., ```Seine & Rat 2016``` for Manning Canal

## Section 1: Setup

## 0. Import required modules

In [1]:
# Sytem
import os
import sys
import fileinput
from pathlib import Path
import shutil
from glob import glob
import tempfile
import subprocess


# Analysis
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint, MultiPolygon, mapping
from shapely.geometry import shape as shapely_shape
from scipy.ndimage import distance_transform_edt, label
import rasterio
import rasterio.features
from rasterio.crs import CRS
from rasterio.mask import mask
from bs4 import BeautifulSoup
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.merge import merge
from rasterio.enums import Resampling
from rasterio.windows import Window
import collections
from collections import defaultdict
# do not use pip to install gdal, use conda instead (conda install -c conda-forge gdal)
from osgeo import gdal


# Visualization
import matplotlib.pyplot as plt
import rasterio.plot


# Whitebox
from WBT.whitebox_tools import WhiteboxTools


sys.path.append("..")
from src.py.proj_conf import *

## 1. Organize data folders and files

In [2]:
class usr_input:
    def string(self, description, default_value=None):
        """
        Prompt the user for a string input with a description.
        If no input is provided, return the default value if specified.
        """
        prompt = f"Enter {description} (default: {default_value}): "
        user_input = input(prompt).strip()
        if user_input == "" and default_value is not None:
            print(f"No {description} provided. Default applied ('{default_value}').")
            return default_value
        elif user_input == "":
            while user_input == "":
                user_input = input(f"A value is required. " + prompt).strip()
        return user_input
    
    def file(self, description, default_value=None):
        """
        Prompt the user for a file name input with a description.
        If no input is provided, return the default value if specified.
        """
        prompt = f"Enter {description} filename (e.g., '{default_value}'). " + \
                "If entering multiple raster files, separate them with commas (e.g., raster1,raster2,raster3) "
        filename = input(prompt)
        if description == "LiDAR DEM raster" and "," in filename:
            filename = [f.strip() for f in filename.split(",")]
            if default_value is None and filename == "":
                while filename == "":
                    filename = input(f"A file name is required. " + prompt).strip()
            elif filename == "":
                filename = default_value
                print(f"No {description} name provided. Default applied ('{default_value}').")
        else:
            filename = filename.strip()
            if "." in filename:
                filename = input("Please do not include file extension in the name. " + prompt).strip()
            elif default_value is None and filename == "":
                while filename == "":
                    filename = input(f"A file name is required. " + prompt).strip()
            elif filename == "":
                filename = default_value
                print(f"No {description} name provided. Default applied ('{default_value}').")
        return filename


# OLD CODE, DO NOT DELETE FOR NOW
# class usr_input:
#     def string(self, description, default_value=None):
#         """
#         Prompt the user for a string input with a description.
#         If no input is provided, return the default value if specified.
#         """
#         prompt = f"Enter {description} (default: {default_value}): "
#         user_input = input(prompt).strip()
#         if user_input == "" and default_value is not None:
#             print(f"No {description} provided. Default applied ('{default_value}').")
#             return default_value
#         elif user_input == "":
#             while user_input == "":
#                 user_input = input(f"A value is required. " + prompt).strip()
#         return user_input
    
#     def file(self, description, default_value=None):
#         """
#         Prompt the user for a file name input with a description.
#         If no input is provided, return the default value if specified.
#         """
#         prompt = f"Enter {description} filename (e.g., '{default_value}'). " + \
#                 "If entering multiple raster files, separate them with commas (e.g., raster1,raster2,raster3) "
#         if description == "LiDAR DEM raster":
#             filename = [f.strip() for f in input(prompt).split(",")]
#         else:
#             filename = input(prompt).strip()
#             if "." in filename:
#                 filename = input("Please do not include file extension in the name. " + prompt).strip()
#             elif default_value is None and filename == "":
#                 while filename == "":
#                     filename = input(f"A file name is required. " + prompt).strip()
#             elif filename == "":
#                 filename = default_value
#                 print(f"No {description} name provided. Default applied ('{default_value}').")
#         return filename

# Usage:
# CLRH_FILENAME = usr_input().file()

### 1.1 Create "DepressionDepths" subfolder

In [3]:
# Set watershed name
WATERSHED_NAME = usr_input().string("watershed", "Manning")


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Specify path for watershed folder
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for DepressionDepths folder
DEPRESSION_DEPTHS_PATH = WATERSHED_PATH + r"/DepressionDepths"


# Create watershed folder in specified path
os.makedirs(DEPRESSION_DEPTHS_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "DepressionDepths" subfolder ("Raw", "Interim", "Processed")

In [4]:
# Specify paths for depression depths subfolders
DD_RAW_PATH = DEPRESSION_DEPTHS_PATH + r"/Raw/"
DD_INTERIM_PATH = DEPRESSION_DEPTHS_PATH + r"/Interim/"
DD_PROCESSED_PATH = DEPRESSION_DEPTHS_PATH + r"/Processed/"


# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [DD_RAW_PATH,
                   DD_INTERIM_PATH,
                   DD_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [5]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(DD_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

## 2. Declare input data file names as variables

In [6]:
# INPUTS
# Name of watershed shapefile from CLRH hydrofabrics zip file (.shp)
CLRH_FILENAME = usr_input().file("hydrofabric shapefile", "finalcat_info_v1-0")


# Name of LiDAR DEM raster from LiDAR DEM zip file (.tif)
# This can be multiple files, separated by commas
# Multiple rasters will be merged into one in latter steps
LIDAR_FILENAME = usr_input().file("LiDAR DEM raster", "sr_dem_cgvd28")


# Name of streams shapefile from NHN streams zip file (.shp)
NHN_FILENAME = usr_input().file("NHN streams shapefile", "NHN_05OE000_5_0_HD_SLWATER_1")

No hydrofabric shapefile name provided. Default applied ('finalcat_info_v1-0').
No LiDAR DEM raster name provided. Default applied ('sr_dem_cgvd28').
No NHN streams shapefile name provided. Default applied ('NHN_05OE000_5_0_HD_SLWATER_1').


In [7]:
# Boolean object to indicate if there are multiple LiDAR DEM rasters
MULTIPLE_LIDAR_RASTERS = True if (isinstance(LIDAR_FILENAME, list) and len(LIDAR_FILENAME)) > 1 else False
print("There are multiple LiDAR rasters: ", MULTIPLE_LIDAR_RASTERS)

There are multiple LiDAR rasters:  False


In [8]:
# Clipped NHN shapefile name with PATH
NHN_CLIPPED_FILE = DD_INTERIM_PATH + \
                  NHN_FILENAME + \
                  "_clip"

## 3. Read CLRH subbasins shapefile

In [17]:
# Load CLRH subbasins shapefile
clrh_gdf = gpd.read_file(DD_RAW_PATH + \
                        CLRH_FILENAME + \
                        ".shp")

## 4. Merge rasters if multiple LiDAR rasters provided

Please note that each cell in this section starts with an if statement to check whether there are multiple input LiDAR raster DEMs since Jupyter does not allow us to break up a huge if block into multiple cells. In a normal .py file, there would only be one if statement with a huge code block. 

### 4.1 Clip each raster to CLRH subbasins shapefile for memory efficiency

In [9]:
# Declare list for storing LiDAR filenames
if MULTIPLE_LIDAR_RASTERS:
    LIDAR_FILENAMES_LIST = [f for f in LIDAR_FILENAME]
    print(LIDAR_FILENAMES_LIST)

In [18]:
# If multiple rasters are provided, clip each of them to subbasins file
if MULTIPLE_LIDAR_RASTERS:
    for file in LIDAR_FILENAMES_LIST:
        # 1. Project shapefile to match raster CRS

        # Open raster file and read its CRS
        with rasterio.open(DD_RAW_PATH + \
                        file + \
                        ".tif") as src:
            input_DEM_crs = src.crs
        
        # Project subbasins data to match raster CRS
        clrh_gdf_projected = clrh_gdf.to_crs(input_DEM_crs)

        # Check if shapefile projection aligns with DEM projection
        is_correctly_projected_clrh = (input_DEM_crs == clrh_gdf_projected.crs)
        

        # 2. Clip raster to watershed boundary

        # Convert projected subbasins data to GeoJSON-like format
        shapes = [mapping(geom) for geom in clrh_gdf_projected.geometry]

        # Mask (clip) the input DEM file
        with rasterio.open(DD_RAW_PATH + \
                            file + \
                            ".tif") as src:
            nodata_value = src.nodata
            out_image, out_transform = mask(src, 
                                            shapes, 
                                            crop=True)
            out_meta = src.meta.copy()

        # Update the copied metadata to match the clipped raster's dimensions and transform
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "compress": "lzw",              # apply LZW compression (for smaller file size)
            "tiled": True,                  # enable tiling (for faster access)
            "blockxsize": 256,              # set block size for tiling (for faster access)
            "blockysize": 256,              # set block size for tiling (for faster access)
            "nodata": nodata_value          # preserve nodata values
        })

        # Clipped DEM file name with path
        LIDAR_CLIPPED_FILE = DD_INTERIM_PATH + \
                                file + \
                                "_clip"

        # Write clipped DEM data into file
        with rasterio.open(LIDAR_CLIPPED_FILE + ".tif",
                        "w", 
                        **out_meta) as dest:
            dest.write(out_image)


### 4.2 Find the CRS for the highest resolution raster

In [11]:
# If multiple rasters are provided, find the target CRS and resolution
if MULTIPLE_LIDAR_RASTERS:
    # Record the paths of all clipped raster files
    clipped_raster_paths = glob(f"{DD_INTERIM_PATH}*_clip.tif")


    # Function to get the resolution of a raster file
    def get_resolution(path):
        with rasterio.open(path) as src:
            return min(abs(src.res[0]), abs(src.res[1]))
        

    # Record the path for the highest-resolution raster
    highest_res_path = min(clipped_raster_paths,
                        key=get_resolution)


    # Record the resolution of the highest-resolution raster
    highest_res_resolution = get_resolution(highest_res_path)


    # Record the CRS for the highest resolution raster
    with rasterio.open(highest_res_path) as ref_src:
        target_crs = ref_src.crs

### 4.3 Project all rasters to the CRS of the highest resolution raster

In [12]:
# If multiple rasters are provided, reproject them to the CRS of the highest-resolution raster
if MULTIPLE_LIDAR_RASTERS:
    # Initialize an empty list to store reprojected raster paths
    reprojected_paths = []

    # Reproject rasters to the crs of the highest-resolution raster
    for path in clipped_raster_paths:

        # Output file name
        out_filename = os.path.splitext(os.path.basename(path))[0] + \
                                "_reprojected.tif"
        # Output file path
        out_path = os.path.join(DD_INTERIM_PATH, out_filename)
        
        with rasterio.open(path) as src:
            # Check if the raster's CRS is different from the target CRS
            if src.crs != target_crs:
                transform, width, height = calculate_default_transform(
                    src.crs, # current raster's CRS
                    target_crs, # target CRS (CRS of the highest-resolution raster)
                    src.width, src.height, *src.bounds # bounds of the current raster
                    )

                # Update metadata for the output raster
                kwargs = src.meta.copy()
                kwargs.update({
                    'crs': target_crs,
                    'transform': transform,
                    'width': width,
                    'height': height,
                    "compress": "lzw",              # apply LZW compression (for smaller file size)
                    "tiled": True,                  # enable tiling (for faster access)
                    "blockxsize": 256,              # set block size for tiling (for faster access)
                    "blockysize": 256               # set block size for tiling (for faster access)
                })

                # Reproject and write the raster
                with rasterio.open(out_path, 'w', **kwargs) as dst:
                    # Loop through each band in the source raster
                    for i in range(1, src.count + 1): 
                        reproject(
                            source=rasterio.band(src, i), # the input band to reproject
                            destination=rasterio.band(dst, i), # where to write the reprojected band
                            src_transform=src.transform, # how to interpret the source raster's coordinates
                            src_crs=src.crs, # source raster's CRS
                            dst_transform=transform, # how to interpret the destination raster's coordinates
                            dst_crs=target_crs, # target CRS
                            resampling=Resampling.bilinear, # resampling method
                            dst_nodata=src.nodata
                        )
                # Append the reprojected raster path to the list
                reprojected_paths.append(out_path)

                # Print confirmation message
                print(f"Raster {path} reprojected to {target_crs} and saved as {out_path}.")
            else:
                # Print message if no reprojection is needed
                print(f"Raster {path} is already in the target CRS. No reprojection needed.")

                # Append the original path to the list of reprojected paths
                reprojected_paths.append(path)

### 4.4 Merge all rasters together

In [13]:
# If multiple rasters are provided, merge them after reprojection
if MULTIPLE_LIDAR_RASTERS:
    # Path for the merged output raster
    LIDAR_FILENAME_NEW = "merged_average_dem"
    out_path_merged = os.path.join(DD_INTERIM_PATH, LIDAR_FILENAME_NEW + ".tif")
    out_path_merged_vrt = os.path.join(DD_INTERIM_PATH, "merged_virtual_raster.vrt")


    # Create a visual raster that includes all input rasters 
    # (virtual rasters are not written to disk, so more memory efficient!)
    vrt = gdal.BuildVRT(out_path_merged_vrt, reprojected_paths)


    # Now use gdal.Warp to resample and average overlapping areas
    gdal.Warp(
        out_path_merged,
        vrt,
        xRes=highest_res_resolution,
        yRes=highest_res_resolution,
        resampleAlg='average', # average elevation for overlapping areas
        options=gdal.WarpOptions(
            format='GTiff',
            creationOptions=['COMPRESS=LZW', 
                            'TILED=YES', 
                            'BLOCKXSIZE=256', 
                            'BLOCKYSIZE=256'] # to limit memory usage and improve performance
        )
    )

    # Close the virtual raster to free up space
    vrt = None

### 4.5 Fill in gaps in merged raster

In [14]:
if MULTIPLE_LIDAR_RASTERS:
    # Path to the merged raster (input for filling nodata values)
    input_raster = out_path_merged
    # Path to the filled output raster
    output_raster = os.path.join(DD_INTERIM_PATH, "merged_average_dem_filled.tif")


    def fill_with_buffer(src, dst, band_index=1, nodata_val=-9999, buffer=20):
        """
        Fill nodata values in a raster using local nearest-neighbor interpolation with a buffer.

        Parameters:
        ------------------------------
        - src: Source rasterio dataset
        - dst: Destination rasterio dataset
        - band_index: Index of the band to process (default is 1)
        - nodata_val: Value representing nodata in the raster (default is -9999)
        - buffer: Size of the buffer around nodata pixels to consider for interpolation (default is 20 pixels)
        """

        # Process block by block (otherwise will run out of memory for large rasters)
        # ji is the block index (row, column) and window is the Window object
        for ji, window in src.block_windows(band_index):
            

            # Extract starting row and column offsets and block dimensions
            row_off, col_off = window.row_off, window.col_off
            height, width = window.height, window.width


            # Define a larger window around the block (otherwise will not be able to fill nodata values at the edges)
            buffered_window = Window(
                max(0, col_off - buffer),
                max(0, row_off - buffer), 
                min(width + 2 * buffer, src.width - col_off + buffer), 
                min(height + 2 * buffer, src.height - row_off + buffer)
            )


            # Read buffered window
            data = src.read(band_index, window=buffered_window)
            mask = data == nodata_val

            # Skip blocks without gaps for efficiency
            if not np.any(mask):
                dst.write(data[buffer:-buffer, buffer:-buffer], band_index, window=window)
                continue


            # Perform local nearest-neighbor interpolation (distance transform gives index of nearest non-nodata pixel)
            inds = distance_transform_edt(mask, 
                                        return_distances=False, 
                                        return_indices=True)
            

            # Make a copy of the data to fill nodata values
            filled = data.copy()


            # Replace nodata values with nearest neighbor values
            filled[mask] = data[tuple(inds[:, mask])]


            # Extract central (original) window and write it (trim back to original block size to avoid overlap)
            filled_central = filled[buffer:buffer+height, buffer:buffer+width]
            dst.write(filled_central, band_index, window=window)

    # Open the input raster and create a new output raster with the same profile
    with rasterio.open(input_raster) as src:
        profile = src.profile.copy()
        nodata = src.nodata if src.nodata is not None else -9999
        profile.update(nodata=nodata)

        # Update the profile for the output raster
        with rasterio.open(output_raster, 'w', **profile) as dst:
            fill_with_buffer(src, dst, band_index=1, nodata_val=nodata, buffer=50)


    # Rename lidar filename variable (to be used in the next steps)
    LIDAR_FILENAME_NEW = LIDAR_FILENAME_NEW + "_filled"

In [15]:
# Change the input LiDAR filename to the merged raster file if multiple rasters were provided
if MULTIPLE_LIDAR_RASTERS:
    # If multiple rasters were provided, use the merged raster file
    LIDAR_FILENAME = LIDAR_FILENAME_NEW 
print(LIDAR_FILENAME)

sr_dem_cgvd28


## 5. Process LiDAR raster DEM

### 5.1 Project subbasins shapefile to align with LiDAR raster DEM

In [19]:
# Open LiDAR DEM and check CRS
LIDAR_ROOT = DD_INTERIM_PATH if MULTIPLE_LIDAR_RASTERS else DD_RAW_PATH
with rasterio.open(LIDAR_ROOT + \
                   LIDAR_FILENAME + \
                   ".tif") as src:
    input_DEM_crs = src.crs


# Project subbasins data to match DEM
clrh_gdf_projected_lidar = clrh_gdf.to_crs(input_DEM_crs)


# Remove any non-alphanumeric characters from crs name
input_DEM_crs_alnum = ''.join(c for c in \
                              str(input_DEM_crs) if c.isalnum())


# Projected subbasins shapefile name with path
CLRH_PROJ_LIDAR_FILE = DD_INTERIM_PATH + \
                        CLRH_FILENAME + \
                        f"_projected_{input_DEM_crs_alnum}"


# Write projected subbasins data to shapefile
clrh_gdf_projected_lidar.to_file(CLRH_PROJ_LIDAR_FILE + ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_clrh_lidar = (input_DEM_crs == clrh_gdf_projected_lidar.crs)
                          

# Print results
print("Shapefile projection is aligned with DEM projection: ", 
      is_correctly_projected_clrh_lidar)
print("\nIf True, proceed to clipping.\n")

Shapefile projection is aligned with DEM projection:  True

If True, proceed to clipping.



c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011982 of field Seg_ID of feature 0 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002444 of field Seg_ID of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002428 of field Seg_ID of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011981 of field Seg_ID of feature 3 not successfully written. Possibly due to too larger number with respect to f

### 5.2 Clip LiDAR raster DEM to watershed

In [20]:
# Only clip it if there is only one LiDAR raster file 
# (if there are multiple rasters, they were already clipped in the previous step)
if not MULTIPLE_LIDAR_RASTERS:
    # Convert projected subbasins data to GeoJSON-like format
    shapes = [mapping(geom) for geom in clrh_gdf_projected_lidar.geometry]


    # Mask (clip) the input DEM file
    with rasterio.open(LIDAR_ROOT + \
                        LIDAR_FILENAME + \
                        ".tif") as src:
        nodata_value = src.nodata
        out_image, out_transform = mask(src, 
                                        shapes, 
                                        crop=True)
        out_meta = src.meta.copy()


    # Update the copied metadata to match the clipped raster's dimensions and transform
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })


    # Clipped DEM file name with path
    LIDAR_CLIPPED_FILE = DD_INTERIM_PATH + \
                            LIDAR_FILENAME + \
                            "_clip"


    # Write clipped DEM data into file
    with rasterio.open(LIDAR_CLIPPED_FILE + ".tif",
                    "w", 
                    **out_meta) as dest:
        dest.write(out_image)


    # Print out some stats fyi
    valid_out_image = out_image[0][out_image[0] != nodata_value]
    print("Clipped unique values:", np.unique(valid_out_image))
    print("Clipped number of unique values: ", len(np.unique(valid_out_image)))
    print("Clipped maximum value :", np.max(valid_out_image))
    print("Clipped minimum value (excluding nodata value): ",  np.min(valid_out_image))
    print("\nIf the numbers make sense, proceed to resampling.\n")
    """ #=======================ELENA CODE (WILL DELETE IT SOON)===================================
    ### SUBBASIN AND STREAM SHAPEFILES WERE PROJECTED IN ARCGIS PRO (NO PROJECTION TOOL IN WHITEBOX)

    # Clip NHN streams to study area
    # wbt.clip(
    #    i = 'NHN_05OE000_5_0_HD_SLWATER_1.shp',
    #    clip = 'clrh_basins_projected_4617.shp',
    #    output = 'NHN_Clip.shp'
    #)

    ### RASTER DATA WAS CLIPPED AND RESAMPLED IN ARCGIS PRO DUE TO MEMORY ALLOCATION ERROR

    # Clip dem raster to study area
    #wbt.clip_raster_to_polygon(
        #i = 'sr_dem_cgvd28.tif',
        #polygons = 'finalcat_info_v1-0_Project.shp',
        #output = 'sr_dem_cgvd28_Clip_MC.tif'
    #)

    # Resample DEM to 5m (for faster processing)
    #wbt.resample(
        #inputs = 'sr_dem_cgvd28_Clip_MC.tif',
        #output = 'sr_dem_cgvd28_Clip_MC_Resample_5m_corr',
        #cell_size = 5,
        #method = "cc"
    #) """
else:
    LIDAR_CLIPPED_FILE = LIDAR_ROOT + LIDAR_FILENAME

Clipped unique values: [236.857 236.861 236.863 ... 317.783 317.788 317.872]
Clipped number of unique values:  634512
Clipped maximum value : 317.872
Clipped minimum value (excluding nodata value):  236.857

If the numbers make sense, proceed to resampling.



### 5.3 Resample clipped LiDAR raster DEM to 5m resolution

In [21]:
# Initialize whitebox tools object
wbt = WhiteboxTools()

In [22]:
# Define resolution
resolution = 5 
resolution_units = "m" # metres

LIDAR_CLIPPED_RESAMPLED_FILE = LIDAR_CLIPPED_FILE + \
                        f"_resample_{resolution}{resolution_units}"

subprocess.run([
    "gdalwarp",
    "-tr", str(resolution), str(resolution),
    "-r", "cubic",
    LIDAR_CLIPPED_FILE + ".tif",
    LIDAR_CLIPPED_RESAMPLED_FILE + ".tif"
])

# Do not use WhiteboxTools resample function (memory allocation error if rasters too large)                               
# print(LIDAR_CLIPPED_RESAMPLED_FILE)
# print(LIDAR_CLIPPED_FILE)
# # Resample to defined resolution
# wbt.resample(
#     inputs=LIDAR_CLIPPED_FILE+".tif",
#     output=LIDAR_CLIPPED_RESAMPLED_FILE+".tif",
#     cell_size=resolution,
#     method="cc"
# )


CompletedProcess(args=['gdalwarp', '-tr', '5', '5', '-r', 'cubic', 'c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/sr_dem_cgvd28_clip.tif', 'c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/sr_dem_cgvd28_clip_resample_5m.tif'], returncode=1)

In [23]:
# Open the clipped resampled file to get some stats
with rasterio.open(LIDAR_CLIPPED_RESAMPLED_FILE + ".tif") as src:
    image = src.read(1) # Read first layer (only has one layer)


# Print out some stats fyi
valid_image = np.ma.masked_equal(image, nodata_value)
print("Clipped resampled DEM unique values:", np.unique(valid_image))
print("Clipped resampled number of unique values: ", len(np.unique(valid_image)))
print("Clipped resampled maximum value :", np.max(valid_image))
print("Clipped resampled minimum value (excluding nodata value): ",  np.min(valid_image))
print("\nIf the numbers make sense, proceed to next sections.")

Clipped resampled DEM unique values: [236.91299438476562 236.91600036621094 236.93899536132812 ...
 316.9700012207031 317.44000244140625 --]
Clipped resampled number of unique values:  290945
Clipped resampled maximum value : 317.44
Clipped resampled minimum value (excluding nodata value):  236.913

If the numbers make sense, proceed to next sections.


## 6. Process NHN Streams Network Shapefile

### 6.1 Project subbasins shapefile to align with NHN streams shapefile

In [24]:
# Load NHN streams shapefile
nhn_gdf = gpd.read_file(DD_RAW_PATH + \
                        NHN_FILENAME + \
                         ".shp")

In [25]:
# CRS for NHN shapefile
input_NHN_crs = nhn_gdf.crs


# Project subbasins data to match streams data
clrh_gdf_projected_nhn = clrh_gdf.to_crs(input_NHN_crs)


# Remove any non-alphanumeric characters from crs name
input_NHN_crs_alnum = ''.join(c for c in \
                              str(input_NHN_crs) if c.isalnum())


# Projected subbasins shapefile name with path
CLRH_PROJ_NHN_FILE = DD_INTERIM_PATH + \
                        CLRH_FILENAME + \
                        f"_projected_{input_NHN_crs_alnum}" 


# Write projected subbasins data to shapefile
clrh_gdf_projected_nhn.to_file(CLRH_PROJ_NHN_FILE + \
                               ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_clrh_nhn = (input_NHN_crs == clrh_gdf_projected_nhn.crs)
                          

# Print results
print("Shapefile projection is aligned with NHN projection: ", 
      is_correctly_projected_clrh_nhn)
print(f"Projection is: {input_NHN_crs}")
print("\nIf True and EPSG:4617 are returned, proceed to clipping.")

Shapefile projection is aligned with NHN projection:  True
Projection is: EPSG:4617

If True and EPSG:4617 are returned, proceed to clipping.


c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011982 of field Seg_ID of feature 0 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002444 of field Seg_ID of feature 1 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310002428 of field Seg_ID of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
c:\Users\iyaktubay\AppData\Local\anaconda3\envs\pwa_dev\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Value 310011981 of field Seg_ID of feature 3 not successfully written. Possibly due to too larger number with respect to f

### 6.2 Clip NHN streams shapefile to watershed

TS Note - Can we adjust this so the depressions raster and the other shapefiles (e.g. hydrofabric) have the same CRS at this point? 

In [26]:

# Clip NHN streams shapefile to watershed
wbt.clip(
   i=DD_RAW_PATH+NHN_FILENAME+".shp",
   clip=CLRH_PROJ_NHN_FILE+".shp",
   output=NHN_CLIPPED_FILE+".shp"
)


# Load NHN streams shapefile
nhn_gdf_clip = gpd.read_file(NHN_CLIPPED_FILE + \
                             ".shp")


# Project clipped NHN shapefile to match DEM
nhn_gdf_clip_projected_lidar = nhn_gdf_clip.to_crs(input_DEM_crs)


# Clipped and projected NHN shapefile name with path
NHN_CLIPPED_PROJECTED_LIDAR_FILE = DD_INTERIM_PATH + \
                                    NHN_FILENAME + \
                                    f"_clip_projected_{input_DEM_crs_alnum}"


# Write clipped and projected NHN shapefile
nhn_gdf_clip_projected_lidar.to_file(NHN_CLIPPED_PROJECTED_LIDAR_FILE + \
                                     ".shp")


# Check if shapefile projection aligns with DEM projection
is_correctly_projected_nhn_lidar = (input_DEM_crs == nhn_gdf_clip_projected_lidar.crs)


# Print results
print("NHN shapefile projection is aligned with DEM projection: ", 
      is_correctly_projected_nhn_lidar)
print(f"Projection is: {input_DEM_crs}")
print("\nIf True and EPSG:26914 are returned, proceed to clipping.")

.\whitebox_tools.exe --run="Clip" --input='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Raw/NHN_05OE000_5_0_HD_SLWATER_1.shp' --clip='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/finalcat_info_v1-0_projected_EPSG4617.shp' --output='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/NHN_05OE000_5_0_HD_SLWATER_1_clip.shp' -v --compress_rasters=False

****************************
* Welcome to Clip          *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading data...
Progress: 0%
Progress: 1%
Progress: 2%
Progress: 3%
Progress: 4%
Progress: 5%
Progress: 6%
Progress: 7%
Progress: 8%
Progress: 9%
Progress: 10%
Progress: 11%
Progress: 12%
Progress: 13%
Progress: 14%
Progress: 15%
Progress: 16%
Progress: 17%
Progress: 18%
Progress: 19%
Progress: 20%
Progress: 21%
Progress: 22%
Progress: 23%
Progress: 24%
Progress: 25%
Progress: 26%
Progress: 27%
Progress: 28%
Progress: 29%
Progress: 30%
Progress: 31%
Progress: 32%
Pr

### 6.3 Visualize clipped and projected NHN streams shapefile

In [ ]:
# Visualize clipped and projected NHN streams shapefile
nhn_gdf_clip_projected_lidar.explore()

## 7. Generate depressions raster and Wetlands Shapefile

### 7.1 Generate depressions raster

In [29]:
# Burn streams into DEM and fill
wbt.fill_burn(
    dem=LIDAR_CLIPPED_RESAMPLED_FILE+".tif",
    streams=NHN_CLIPPED_PROJECTED_LIDAR_FILE+".shp",
    output=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn"+".tif"
)


# Subtract raw DEM from filled DEM
first_statement_operand = LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn"+".tif"
second_statement_operand = LIDAR_CLIPPED_RESAMPLED_FILE+".tif"
wbt.raster_calculator(
    output = LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif",
    statement = f"'{first_statement_operand}' - '{second_statement_operand}'"  
)

# Depressions raster file with path
DEPRESSIONS_RASTER_TEMP_FILE = DD_INTERIM_PATH+LIDAR_FILENAME+\
                        f"_clip_resample_{resolution}{resolution_units}_FillBurn_Deps_Corr_temp"
# Remove stray burn lines
wbt.conditional_evaluation(
    i=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif",
    output=DEPRESSIONS_RASTER_TEMP_FILE+".tif",
    statement="value < 0.0",
    true=0.0,
    false=LIDAR_CLIPPED_RESAMPLED_FILE+"_FillBurn_Deps"+".tif"
)



.\whitebox_tools.exe --run="FillBurn" --dem='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/sr_dem_cgvd28_clip_resample_5m.tif' --streams='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/NHN_05OE000_5_0_HD_SLWATER_1_clip_projected_EPSG26914.shp' --output='c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Interim/sr_dem_cgvd28_clip_resample_5m_FillBurn.tif' -v --compress_rasters=False

****************************
* Welcome to FillBurn      *
* Powered by WhiteboxTools *
* www.whiteboxgeo.com      *
****************************
Reading streams data...
Reading DEM data...
Rasterizing Streams: 0%
Rasterizing Streams: 1%
Rasterizing Streams: 2%
Rasterizing Streams: 3%
Rasterizing Streams: 4%
Rasterizing Streams: 5%
Rasterizing Streams: 6%
Rasterizing Streams: 7%
Rasterizing Streams: 8%
Rasterizing Streams: 9%
Rasterizing Streams: 10%
Rasterizing Streams: 11%
Rasterizing Streams: 12%
Rasterizing Streams: 13%
Rasterizing Streams: 14%
Rasterizing Streams: 15%
Ras

0

In [30]:
# Path for depressions raster with adjusted CRS
DEPRESSIONS_RASTER_FILE = DD_PROCESSED_PATH+LIDAR_FILENAME+\
                        f"_clip_resample_{resolution}{resolution_units}_FillBurn_Deps_Corr"

with rasterio.open(DEPRESSIONS_RASTER_TEMP_FILE + ".tif") as src:
    profile = src.profile
    profile.update(crs=CRS.from_epsg(input_DEM_crs_alnum[4:]))  # Example EPSG

    with rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif", "w", **profile) as dst:
        dst.write(src.read())

### 7.2 Generate Wetlands Shapefile with stats (this is optional for the user and not needed for RAVEN)

In [31]:
# Ask user if they want to generate a wetlands shapefile with stats
WANT_WETLANDS_SHAPEFILE = usr_input().string("\"Yes\" if you would like to also generate a wetlands shapefile with stats (NOT required for RAVEN)", "No")

In [32]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":
    # Load the depression depths raster
    with rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif") as src:
        depression_data = src.read(1)  # Read first layer (only has one layer)
        transform = src.transform
        pixel_area = abs(transform[0] * transform[4])
        nodata = src.nodata

In [ ]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":    
    # Mask nodata and select depressions
    depth_threshold = 0.05  # ignore anything below 5 cm (for efficiency later)
    valid_mask = (depression_data > depth_threshold) & (depression_data != nodata)

In [ ]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":
    # Label all connected depressions > 0 (wetlands)
    structure = np.ones((3, 3), dtype=int)
    labeled_array, num_features = label(valid_mask, structure=structure)


    # Flatten arrays
    flat_labels = labeled_array.ravel()
    flat_depths = depression_data.ravel()


    # Get pixel counts per label (per wetland)
    counts = np.bincount(flat_labels)
    areas_m2 = counts * pixel_area


    # Total storage volume per label (per wetland)
    volume_sums = np.bincount(flat_labels, weights=flat_depths)
    volumes_m3 = volume_sums * pixel_area


    # Preallocate median array (this is so that we can avoid for loops)
    medians = np.full(num_features + 1, np.nan)

In [ ]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":
    depths_by_label = defaultdict(list)
    for label_val, depth in zip(flat_labels, flat_depths):
        if label_val == 0:
            continue
        depths_by_label[label_val].append(depth)

    for label_val, depth_list in depths_by_label.items():
        medians[label_val] = np.median(depth_list)


    # Build dataframe
    wetlands_df_stats = pd.DataFrame({
        'wetland_id': np.arange(len(areas_m2)),
        'area_m2': areas_m2,
        'volume_m3': volumes_m3,
        'median_depth_m': medians
    })


    # Filter out background (label 0), small areas, and small volumes
    area_threshold = 100  # m2
    volume_threshold = 30  # m3
    wetlands_df_stats = wetlands_df_stats.query(f"wetland_id != 0 and area_m2 >= {area_threshold} and volume_m3 >= {volume_threshold}")


    # Write stats to CSV
    WETLANDS_STATS_CSV = DD_PROCESSED_PATH + "Wetlands_Stats.csv"
    wetlands_df_stats.to_csv(WETLANDS_STATS_CSV, index=False) 


In [ ]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":
    # Create mask of valid wetland IDs (from filtered df_stats)
    valid_ids = set(wetlands_df_stats["wetland_id"])
    valid_mask = np.isin(labeled_array, list(valid_ids))


    # Extract shapes (polygons) from the labeled array
    polygons = []
    labels = []


    with rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif") as src:  # or any raster for transform/CRS
        transform = src.transform
        crs = src.crs

        for geom, val in rasterio.features.shapes(
            labeled_array.astype(np.int32),
            mask=valid_mask,
            transform=transform
        ):
            if val in valid_ids:
                polygons.append(shapely_shape(geom))
                labels.append(val)

In [ ]:
if WANT_WETLANDS_SHAPEFILE.lower() == "yes":
    # Make GeoDataFrame
    gdf = gpd.GeoDataFrame({'wetland_id': labels, 'geometry': polygons}, crs=crs)


    # Join with statistics table
    gdf = gdf.merge(wetlands_df_stats, on='wetland_id')


    # Write results to shapefile
    WETLANDS_POLYGONS_SHAPEFILE = DD_PROCESSED_PATH + "Wetlands_Polygons_with_Stats.shp"
    gdf.to_file(WETLANDS_POLYGONS_SHAPEFILE)

## 8. Calculate Depression Depths

### 8.1 Calculate Subbasin Zonal Statistics

Note on zonal statistics generated with Whitebox Tools - [link to manual](https://www.whiteboxgeo.com/manual/wbt_book/available_tools/mathand_stats_tools.html#ZonalStatistics)

- It is unclear whether the calculation for "Total" accounts for the pixel resolution, or simply adds the values across all of the pixels. Based on a quick review of the raster and shapefile in ArcGIS, it seems to be the latter (pixel, not square meters). A factor of 25 needs to be added when calculating depression storage for Raven.

In [ ]:
# CLRH subbasins raster file name with path 
CLRH_RASTER_FILE = CLRH_PROJ_LIDAR_FILE+"_raster"


# # Convert clrh subbasin polygons to raster
wbt.vector_polygons_to_raster(
    i=CLRH_PROJ_LIDAR_FILE+".shp",
    output=CLRH_RASTER_FILE+".tif",
    field = "FID",
    nodata = True,
    cell_size = 5.0
)


# Open clrh subbasins raster file
with rasterio.open(CLRH_RASTER_FILE+".tif") as src:
    print(f"CLRH Raster CRS before update: {src.crs}")
    target_crs = input_DEM_crs_alnum[4:]
    profile = src.profile
    profile.update(crs=CRS.from_epsg(target_crs))
    # src.crs = CRS.from_epsg(target_crs)
    print(f"CLRH Raster CRS after update: {src.crs}, profile crs: {profile['crs']}")

    with rasterio.open(CLRH_RASTER_FILE+"_clean.tif", "w", **profile) as dst:
        dst.write(src.read())


CLRH_RASTER_FILE = CLRH_RASTER_FILE+"_clean"

# Zonal stats file name with path
ZONAL_STATS_FILE = Path(DD_PROCESSED_PATH, 
                   f"ZonalStats_{WATERSHED_NAME}.html").resolve()


# Make sure that the hydrofabric raster is aligned with the depression raster
subprocess.run([
    "gdalwarp",
    "-r", "near",               
    "-overwrite",    
    "-tr", "5", "5",            
    "-te",                     
    str(rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif").bounds.left),
    str(rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif").bounds.bottom),
    str(rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif").bounds.right),
    str(rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif").bounds.top),
    CLRH_RASTER_FILE + ".tif",
    CLRH_RASTER_FILE + "_aligned" + ".tif"
])


# Calculate zonal statistics for depression raster
wbt.zonal_statistics(
    i=DEPRESSIONS_RASTER_FILE + ".tif",
    features=CLRH_RASTER_FILE + "_aligned" + ".tif",
    stat="total",
    out_table=ZONAL_STATS_FILE
)

### 8.2 Calculate depression depths from zonal statistics

In [34]:
# Read zonal stats html as pandas df
zonal_stats_df = pd.read_html(ZONAL_STATS_FILE,
                              flavor='bs4')[0]

# Calculate depression depths (mm) and add field to gdf
clrh_gdf_projected_lidar['Deps_Depth_mm'] = zonal_stats_df['Mean'] * 1000

# Calculate depression volumes (m3) and add field to gdf
clrh_gdf_projected_lidar['Deps_Vol_m3'] = clrh_gdf_projected_lidar['Deps_Depth_mm']*clrh_gdf_projected_lidar['BasArea']

# IY code below. TS simplified method above (removes need for Deps_SUM field in output dataset)

# # Join zonal stats totals to clrh subbasins geodataframe
# clrh_gdf_projected_lidar['Deps_SUM'] = zonal_stats_df['Total']


# # Calculate depression volumes (m3) and add field to gdf
# clrh_gdf_projected_lidar['Deps_Vol_m3'] = clrh_gdf_projected_lidar['Deps_SUM']*5*5


# # Calculate depression depths (mm) and add field to gdf
# clrh_gdf_projected_lidar['Deps_Depth_mm'] = clrh_gdf_projected_lidar['Deps_Vol_m3']/ \
#                                             clrh_gdf_projected_lidar['BasArea']*1000

### 8.3 Write depression depths to file

In [ ]:
# Depression depths file name with path
DEPRESSION_DEPTHS_FILE = DD_PROCESSED_PATH + \
                         "CLRH_basins_depression_depths"


# Write geodataframe to file for use in raven input file creation
clrh_gdf_projected_lidar.to_file(DEPRESSION_DEPTHS_FILE + ".shp")

## 9. Visualize Processed Data

In [ ]:
# Open the CLRH raster file
with rasterio.open(CLRH_RASTER_FILE + "_aligned" + ".tif") as src:
    raster = src.read(1)
    nodata = src.nodata
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    print(f"CLRH Raster CRS: {src.crs}")


# Mask no-data values
raster = np.ma.masked_equal(raster, nodata)


# Open depression depths raster file
with rasterio.open(DEPRESSIONS_RASTER_FILE + ".tif") as src:
    depress = src.read(1)
    depress_nodata = src.nodata
    depress_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    print(f"Depressions Raster CRS: {src.crs}")


# Mask no data values
depress = np.ma.masked_equal(depress, depress_nodata)


# Plot
fig, ax = plt.subplots(figsize=(10, 8))


# Plot the raster in grayscale
raster_img = ax.imshow(raster,
                    cmap='gray', 
                    extent=extent,
                    interpolation='nearest')


# Depression raster (overlaid in red or blue with transparency)
ax.imshow(depress, 
          cmap='Blues', 
          extent=depress_extent, 
          alpha=0.5) 


# Overlay the shapefile
nhn_gdf_clip_projected_lidar.plot(ax=ax, 
                                  facecolor='black', 
                                  edgecolor='black', 
                                  linewidth=1)


# Final touches
plt.title(f"{WATERSHED_NAME} Depression Depths")
plt.axis('off')
plt.show()

## 10. Calculate basin aspect ratios from geometry

In [ ]:
# Read the processed depression depths shapefile
clrh_gdf_processed = gpd.read_file(DEPRESSION_DEPTHS_FILE + ".shp")

c:/Users/iyaktubay/PWA/Data/Manning/DepressionDepths/Processed/CLRH_basins_depression_depths.shp


In [37]:
def calculate_aspect_ratio_bbox(geom):
    # Handle MultiPolygon and Polygon
    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        return None

    # Step 1: Convex hull
    hull = geom.convex_hull
    coords = np.array(hull.exterior.coords[:-1])  # Remove duplicate last point

    min_area = None
    best_width = None
    best_height = None

    # Step 2: Iterate through each edge of the convex hull
    for i in range(len(coords)):
        p1 = coords[i]
        p2 = coords[(i + 1) % len(coords)]
        edge_vec = p2 - p1
        edge_length = np.linalg.norm(edge_vec)
        if edge_length == 0:
            continue

        # Calculate rotation matrix to align edge with x-axis
        angle = np.arctan2(edge_vec[1], edge_vec[0])
        cos_angle = np.cos(-angle)
        sin_angle = np.sin(-angle)
        rotation_matrix = np.array([[cos_angle, -sin_angle], [sin_angle, cos_angle]])

        # Step 2a: Project all hull vertices onto rotated axes
        rotated = np.dot(coords - p1, rotation_matrix)

        # Step 2b: Find min/max on rotated axes
        min_x, max_x = rotated[:, 0].min(), rotated[:, 0].max()
        min_y, max_y = rotated[:, 1].min(), rotated[:, 1].max()
        width = max_x - min_x
        height = max_y - min_y
        area = width * height

        # Step 2c: Retain smallest area
        if (min_area is None) or (area < min_area):
            min_area = area
            best_width = width
            best_height = height

    # Step 3: Calculate aspect ratio
    if best_width is None or best_height is None or min(best_width, best_height) == 0:
        return None
    
    # Sanity check of the results
    if min_area < hull.area:
        print("Error: Minimum bounding box area is smaller than convex hull area.")
        return None
    
    aspect_ratio = max(best_width, best_height) / min(best_width, best_height)
    return aspect_ratio

clrh_gdf_processed['Aspect_Ratio'] = clrh_gdf_processed['geometry'].apply(calculate_aspect_ratio_bbox)

In [ ]:
# Write subId to aspect ratio info to CSV (to be used in spotpy later...)
clrh_gdf_processed[["SubId", "Aspect_Ratio"]].to_csv(f"{DD_PROCESSED_PATH}/aspect_ratios.csv", index=False)